In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Busca la raíz del proyecto desde la ubicación del notebook
ROOT = Path.cwd()

while ROOT != ROOT.parent and not (ROOT / "database").exists():
    ROOT = ROOT.parent

DB_PATH = ROOT / "database" / "air_quality.db"

print("Ruta de la base de datos:", DB_PATH)
print("¿La base existe?:", DB_PATH.exists())

conexion = sqlite3.connect(DB_PATH)

Ruta de la base de datos: /workspaces/etl-calidad-aire-india/database/air_quality.db
¿La base existe?: True


In [2]:
consulta = """
SELECT
    f.measurement_id,
    c.city_name AS City,
    d.full_datetime AS Datetime,
    f.pm25 AS "PM2.5",
    f.pm10 AS PM10,
    f.no_value AS NO,
    f.no2 AS NO2,
    f.nox AS NOx,
    f.nh3 AS NH3,
    f.co AS CO,
    f.so2 AS SO2,
    f.o3 AS O3,
    f.aqi AS AQI,
    a.aqi_bucket AS AQI_Bucket,
    d.year AS Year,
    d.month AS Month,
    d.day AS Day,
    d.quarter AS Quarter,
    d.day_of_week AS Day_of_Week
FROM fact_air_quality AS f
INNER JOIN dim_city AS c
    ON f.city_id = c.city_id
INNER JOIN dim_date AS d
    ON f.date_id = d.date_id
INNER JOIN dim_aqi AS a
    ON f.aqi_id = a.aqi_id
ORDER BY f.measurement_id
"""

df_bd = pd.read_sql_query(consulta, conexion)
df_bd["Datetime"] = pd.to_datetime(df_bd["Datetime"])

print("Dimensiones de los datos consultados desde SQLite:", df_bd.shape)

df_bd.head()

Dimensiones de los datos consultados desde SQLite: (18265, 19)


,measurement_id,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket,Year,Month,Day,Quarter,Day_of_Week
0,1,Delhi,2015-01-01,153.3,241.7,182.9,33.0,81.3,38.5,1.87,64.5,83.6,325.8,Very Poor,2015,1,1,1,Thursday
1,2,Mumbai,2015-01-01,70.5,312.7,195.0,42.0,122.5,31.5,7.22,83.8,108.0,262.7,Poor,2015,1,1,1,Thursday
2,3,Chennai,2015-01-01,174.1,275.4,56.2,68.8,230.9,28.5,8.56,60.8,43.9,341.8,Very Poor,2015,1,1,1,Thursday
3,4,Kolkata,2015-01-01,477.2,543.9,14.1,76.4,225.9,45.6,2.41,42.1,171.1,206.3,Poor,2015,1,1,1,Thursday
4,5,Bangalore,2015-01-01,171.6,117.7,123.3,12.4,61.9,49.7,1.26,79.7,164.3,339.8,Very Poor,2015,1,1,1,Thursday
